# AITeamVN Vietnamese Embedding v2 FashionCLIP Projection Training on 336K Samples

This notebook trains a projection/adapter for a fashion-domain Multimodal RAG system:

- Teacher: frozen `patrickjohncyh/fashion-clip` text encoder.
- Student backbone: `AITeamVN/Vietnamese_Embedding_v2`.
- Stage 1: freeze the entire student encoder and train only the projection head.
- Stage 2: unfreeze a few final student encoder layers for light fine-tuning.
- Main trainable module: a projection head that maps Vietnamese embeddings into the FashionCLIP space.
- Data: `fashion_description_336k_cleaned.csv` with EN-VI captions.
- Loss: weighted alignment MSE + structure-preserving loss + contrastive loss.

The goal is to improve fashion-domain image-text retrieval while reducing the risk of damaging the general language knowledge in the student encoder.


In [ ]:
import os
import math
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, CLIPTextModelWithProjection, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

try:
    from transformers import CLIPVisionModelWithProjection, CLIPProcessor
    from PIL import Image
    HAS_VISION_DEPS = True
except Exception:
    HAS_VISION_DEPS = False

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
USE_DATA_PARALLEL = GPU_COUNT > 1

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

print('Device:', device)
print('GPU count:', GPU_COUNT)
if torch.cuda.is_available():
    for gpu_idx in range(GPU_COUNT):
        print(f'GPU {gpu_idx}:', torch.cuda.get_device_name(gpu_idx))
print('Use DataParallel:', USE_DATA_PARALLEL)

In [ ]:
# =========================
# CONFIG
# =========================
DATA_PATH = '/kaggle/input/datasets/qucvinhchu/fashion-description-bilingual-336k/fashion_description_336k_cleaned.csv'
# Nếu upload file theo Kaggle Dataset khác, sửa DATA_PATH tại đây.

TEACHER_MODEL_NAME = 'patrickjohncyh/fashion-clip'
STUDENT_MODEL_NAME = 'AITeamVN/Vietnamese_Embedding_v2'

SAVE_DIR = Path('/kaggle/working/vifashionclip_aiteamvn_embedding_v2_projection_336k')
CACHE_DIR = SAVE_DIR / 'cache'
CACHE_VERSION = 'raw_vi_aiteamvn_v2'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# AITeamVN/Vietnamese_Embedding_v2 uses a BGE-M3/XLM-R style backbone, so no Vietnamese word segmentation is required.

TEACHER_MAX_LENGTH = 77
STUDENT_MAX_LENGTH = 128
PRECOMPUTE_BATCH_SIZE = 512  # DataParallel chia khoảng 256 mẫu/GPU trên 2x T4
TRAIN_BATCH_SIZE = 512       # Projection training nhẹ, batch lớn giúp structure loss ổn hơn
EVAL_BATCH_SIZE = 512

EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
WARMUP_RATIO = 0.06
EARLY_STOPPING_PATIENCE = 4

LAMBDA_ALIGN = 48.0
BETA_STRUCT = 1.0
GAMMA_CONTRASTIVE = 1.0
CONTRASTIVE_TEMPERATURE = 0.07

# Projection head capacity
PROJECTION_HIDDEN_DIM = 1024
PROJECTION_NUM_LAYERS = 3
PROJECTION_DROPOUT = 0.05

# Data quality filters
MIN_EN_WORDS = 5
MIN_VI_WORDS = 5
MAX_EN_WORDS = 160
MAX_VI_WORDS = 120
MIN_RATIO_KEEP = 0.25
MAX_RATIO_KEEP = 2.0

# ratio >= 0.50: full weight; 0.35-0.50: medium; 0.25-0.35: low
RATIO_FULL_WEIGHT = 0.50
RATIO_MEDIUM_WEIGHT = 0.35

USE_AMP = torch.cuda.is_available()
# TensorDataset đã nằm trong RAM, nên không cần multiprocessing workers.
# Trên Kaggle/Jupyter, num_workers > 0 đôi khi gây lỗi cleanup: "can only test a child process".
NUM_WORKERS = 0

print('Save dir:', SAVE_DIR)


## Data Preparation

This step loads the EN-VI dataset, removes duplicates, filters captions that are too short or too long, and assigns lower weights to EN-VI pairs that may contain missing or mismatched information.

Because `AITeamVN/Vietnamese_Embedding_v2` is fine-tuned from a BGE-M3/XLM-R style backbone, this notebook uses raw Vietnamese captions directly and does not apply Vietnamese word segmentation.


In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
df = df[['description_en', 'description_vi']].dropna()
df['description_en'] = df['description_en'].astype(str).str.strip()
df['description_vi'] = df['description_vi'].astype(str).str.strip()

print('Raw rows:', len(df))
duplicate_count = df.duplicated(['description_en', 'description_vi']).sum()
df = df.drop_duplicates(['description_en', 'description_vi']).reset_index(drop=True)
print('Duplicate pairs removed:', duplicate_count)

df['en_words'] = df['description_en'].str.split().str.len()
df['vi_words'] = df['description_vi'].str.split().str.len()
df['ratio_vi_en'] = (df['vi_words'] + 1) / (df['en_words'] + 1)

keep_mask = (
    (df['en_words'] >= MIN_EN_WORDS) &
    (df['vi_words'] >= MIN_VI_WORDS) &
    (df['en_words'] <= MAX_EN_WORDS) &
    (df['vi_words'] <= MAX_VI_WORDS) &
    (df['ratio_vi_en'] >= MIN_RATIO_KEEP) &
    (df['ratio_vi_en'] <= MAX_RATIO_KEEP)
)
df = df[keep_mask].copy().reset_index(drop=True)

df['quality_weight'] = 1.0
df.loc[df['ratio_vi_en'] < RATIO_FULL_WEIGHT, 'quality_weight'] = 0.6
df.loc[df['ratio_vi_en'] < RATIO_MEDIUM_WEIGHT, 'quality_weight'] = 0.3

# No word segmentation for AITeamVN/Vietnamese_Embedding_v2. Use raw Vietnamese captions directly.
print('Clean rows:', len(df))
print(df[['en_words', 'vi_words', 'ratio_vi_en', 'quality_weight']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))
df[['description_vi']].head(3)


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.10, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, shuffle=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Train:', len(train_df))
print('Val:', len(val_df))
print('Test:', len(test_df))

train_df.to_csv(SAVE_DIR / 'train_clean.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(SAVE_DIR / 'val_clean.csv', index=False, encoding='utf-8-sig')
test_df.to_csv(SAVE_DIR / 'test_clean.csv', index=False, encoding='utf-8-sig')


## Stage 1 - Load Teacher and Student Encoder

In Stage 1, both the FashionCLIP text encoder and `AITeamVN/Vietnamese_Embedding_v2` are frozen. These encoders are used only to produce embeddings; the projection head is trained in the following cells.


In [ ]:
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
teacher_model = CLIPTextModelWithProjection.from_pretrained(TEACHER_MODEL_NAME)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad = False

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME)
student_encoder = AutoModel.from_pretrained(STUDENT_MODEL_NAME)
student_encoder.eval()
for p in student_encoder.parameters():
    p.requires_grad = False

STUDENT_DIM = student_encoder.config.hidden_size
TEACHER_DIM = teacher_model.config.projection_dim

teacher_model = teacher_model.to(device)
student_encoder = student_encoder.to(device)
if USE_DATA_PARALLEL:
    teacher_model = nn.DataParallel(teacher_model)
    student_encoder = nn.DataParallel(student_encoder)
    print(f'DataParallel enabled for frozen encoders on {GPU_COUNT} GPUs')

print('Student dim:', STUDENT_DIM)
print('Teacher/FashionCLIP dim:', TEACHER_DIM)


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom

@torch.no_grad()
def encode_teacher_en(texts, batch_size=PRECOMPUTE_BATCH_SIZE):
    embeds = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Teacher EN'):
        batch = texts[i:i + batch_size]
        inputs = teacher_tokenizer(batch, padding=True, truncation=True, max_length=TEACHER_MAX_LENGTH, return_tensors='pt').to(device)
        out = teacher_model(**inputs).text_embeds
        out = F.normalize(out, p=2, dim=-1)
        embeds.append(out.detach().float().cpu())
    return torch.cat(embeds, dim=0)

@torch.no_grad()
def encode_student_vi_base(texts, batch_size=PRECOMPUTE_BATCH_SIZE):
    embeds = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Student VI base'):
        batch = texts[i:i + batch_size]
        inputs = student_tokenizer(batch, padding=True, truncation=True, max_length=STUDENT_MAX_LENGTH, return_tensors='pt').to(device)
        out = student_encoder(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        pooled = mean_pool(out.last_hidden_state, inputs['attention_mask'])
        embeds.append(pooled.detach().float().cpu())
    return torch.cat(embeds, dim=0)


## Stage 1 - Precompute Embeddings

Because both teacher and student encoders are frozen in Stage 1, their embeddings are cached to disk. If the notebook is rerun, this cell loads the cache instead of forwarding through the encoders again.


In [ ]:
def build_or_load_split_cache(split_name, split_df):
    student_path = CACHE_DIR / f'{split_name}_{CACHE_VERSION}_student_vi_base.pt'
    teacher_path = CACHE_DIR / f'{split_name}_{CACHE_VERSION}_teacher_en.pt'
    weight_path = CACHE_DIR / f'{split_name}_{CACHE_VERSION}_weights.pt'

    if student_path.exists() and teacher_path.exists() and weight_path.exists():
        print(f'Loading cache for {split_name}')
        student_base = torch.load(student_path, map_location='cpu')
        teacher = torch.load(teacher_path, map_location='cpu')
        weights = torch.load(weight_path, map_location='cpu')
        return student_base, teacher, weights

    print(f'Building cache for {split_name}')
    en_texts = split_df['description_en'].tolist()
    vi_texts = split_df['description_vi'].tolist()
    weights = torch.tensor(split_df['quality_weight'].values, dtype=torch.float32)

    teacher = encode_teacher_en(en_texts)
    student_base = encode_student_vi_base(vi_texts)

    torch.save(student_base, student_path)
    torch.save(teacher, teacher_path)
    torch.save(weights, weight_path)
    return student_base, teacher, weights

train_student_base, train_teacher, train_weights = build_or_load_split_cache('train', train_df)
val_student_base, val_teacher, val_weights = build_or_load_split_cache('val', val_df)
test_student_base, test_teacher, test_weights = build_or_load_split_cache('test', test_df)

teacher_model.cpu()
student_encoder.cpu()
torch.cuda.empty_cache()
gc.collect()

print('Train tensors:', train_student_base.shape, train_teacher.shape, train_weights.shape)
print('Val tensors:', val_student_base.shape, val_teacher.shape, val_weights.shape)


## Stage 1 - Projection Head and Loss

The projection head is the main trainable component in Stage 1. The loss combines weighted alignment MSE, structure-preserving loss, and contrastive loss to align embeddings and improve retrieval ranking.


In [ ]:
class ResidualMLPBlock(nn.Module):
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


class ProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=1024, num_layers=3, dropout=0.05):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ]
        for _ in range(max(0, num_layers - 2)):
            layers.append(ResidualMLPBlock(hidden_dim, dropout=dropout))
        layers.extend([
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),
        ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def weighted_alignment_loss(student_embeds, teacher_embeds, weights):
    per_sample = F.mse_loss(student_embeds, teacher_embeds, reduction='none').mean(dim=1)
    return (per_sample * weights).sum() / torch.clamp(weights.sum(), min=1e-6)


def structure_preserving_loss(student_embeds, teacher_embeds):
    if student_embeds.size(0) < 2:
        return student_embeds.new_tensor(0.0)
    r_student = student_embeds @ student_embeds.T
    r_teacher = teacher_embeds @ teacher_embeds.T
    mask = torch.triu(torch.ones_like(r_teacher, dtype=torch.bool), diagonal=1)
    return F.mse_loss(r_student[mask], r_teacher[mask])


def contrastive_loss(student_embeds, teacher_embeds, temperature=CONTRASTIVE_TEMPERATURE):
    logits = student_embeds @ teacher_embeds.T
    logits = logits / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_t2i = F.cross_entropy(logits, labels)
    loss_i2t = F.cross_entropy(logits.T, labels)
    return 0.5 * (loss_t2i + loss_i2t)


def total_loss_fn(projected, teacher_embeds, weights):
    projected = F.normalize(projected, p=2, dim=-1)
    teacher_embeds = F.normalize(teacher_embeds, p=2, dim=-1)
    align = weighted_alignment_loss(projected, teacher_embeds, weights)
    struct = structure_preserving_loss(projected, teacher_embeds)
    contr = contrastive_loss(projected, teacher_embeds)
    total = LAMBDA_ALIGN * align + BETA_STRUCT * struct + GAMMA_CONTRASTIVE * contr
    return total, align.detach(), struct.detach(), contr.detach()


projection = ProjectionHead(
    STUDENT_DIM,
    TEACHER_DIM,
    hidden_dim=PROJECTION_HIDDEN_DIM,
    num_layers=PROJECTION_NUM_LAYERS,
    dropout=PROJECTION_DROPOUT,
).to(device)
if USE_DATA_PARALLEL:
    projection = nn.DataParallel(projection)
    print(f'DataParallel enabled for projection head on {GPU_COUNT} GPUs')

print('Trainable params:', sum(p.numel() for p in projection.parameters() if p.requires_grad))


In [ ]:
train_ds = TensorDataset(train_student_base, train_teacher, train_weights)
val_ds = TensorDataset(val_student_base, val_teacher, val_weights)
test_ds = TensorDataset(test_student_base, test_teacher, test_weights)

train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

optimizer = AdamW(projection.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
num_training_steps = EPOCHS * len(train_loader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print('Train batches:', len(train_loader))
print('Warmup steps:', num_warmup_steps, '/', num_training_steps)


## Stage 1 - Train Projection Head

This cell updates only the projection head and does not update the student encoder. This is the safest stage for learning the mapping into FashionCLIP space while preserving the encoder backbone.


In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total, total_align, total_struct, total_contr, n = 0.0, 0.0, 0.0, 0.0, 0
    for student_base, teacher_embeds, weights in loader:
        student_base = student_base.to(device, non_blocking=True)
        teacher_embeds = teacher_embeds.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)
        projected = model(student_base)
        loss, align, struct, contr = total_loss_fn(projected, teacher_embeds, weights)
        bs = student_base.size(0)
        total += loss.item() * bs
        total_align += align.item() * bs
        total_struct += struct.item() * bs
        total_contr += contr.item() * bs
        n += bs
    return {'loss': total / n, 'align': total_align / n, 'struct': total_struct / n, 'contrastive': total_contr / n}


best_val = float('inf')
patience = 0
history = []
best_path = SAVE_DIR / 'best_projection_head.pt'

for epoch in range(1, EPOCHS + 1):
    projection.train()
    running_loss, running_align, running_struct, running_contr, seen = 0.0, 0.0, 0.0, 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')

    for student_base, teacher_embeds, weights in pbar:
        student_base = student_base.to(device, non_blocking=True)
        teacher_embeds = teacher_embeds.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            projected = projection(student_base)
            loss, align, struct, contr = total_loss_fn(projected, teacher_embeds, weights)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(projection.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        bs = student_base.size(0)
        running_loss += loss.item() * bs
        running_align += align.item() * bs
        running_struct += struct.item() * bs
        running_contr += contr.item() * bs
        seen += bs
        pbar.set_postfix(
            loss=running_loss / seen,
            align=running_align / seen,
            struct=running_struct / seen,
            contr=running_contr / seen,
        )

    train_metrics = {
        'loss': running_loss / seen,
        'align': running_align / seen,
        'struct': running_struct / seen,
        'contrastive': running_contr / seen,
    }
    val_metrics = evaluate_loss(projection, val_loader)
    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})
    pd.DataFrame(history).to_csv(SAVE_DIR / 'training_history.csv', index=False)

    print(
        f"Epoch {epoch}: train_loss={train_metrics['loss']:.5f} "
        f"val_loss={val_metrics['loss']:.5f} "
        f"val_align={val_metrics['align']:.6f} "
        f"val_struct={val_metrics['struct']:.6f} "
        f"val_contr={val_metrics['contrastive']:.5f}"
    )

    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        patience = 0
        torch.save({
            'projection_state_dict': unwrap_model(projection).state_dict(),
            'student_model_name': STUDENT_MODEL_NAME,
            'teacher_model_name': TEACHER_MODEL_NAME,
            'student_dim': STUDENT_DIM,
            'teacher_dim': TEACHER_DIM,
            'config': {
                'lambda_align': LAMBDA_ALIGN,
                'beta_struct': BETA_STRUCT,
                'gamma_contrastive': GAMMA_CONTRASTIVE,
                'contrastive_temperature': CONTRASTIVE_TEMPERATURE,
                'projection_hidden_dim': PROJECTION_HIDDEN_DIM,
                'projection_num_layers': PROJECTION_NUM_LAYERS,
                'projection_dropout': PROJECTION_DROPOUT,
                'learning_rate': LEARNING_RATE,
                'train_batch_size': TRAIN_BATCH_SIZE,
            },
        }, best_path)
        print('Saved best checkpoint:', best_path)
    else:
        patience += 1
        print(f'Early stopping counter: {patience}/{EARLY_STOPPING_PATIENCE}')
        if patience >= EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break

print('Best val loss:', best_val)


## Stage 1 - Test Loss and Checkpoint Saving

After Stage 1 finishes, the notebook reloads the best projection checkpoint, computes test loss, and saves the artifacts needed to reuse the projection head.


In [ ]:
checkpoint = torch.load(best_path, map_location=device)
unwrap_model(projection).load_state_dict(checkpoint['projection_state_dict'])
projection.to(device)

test_metrics = evaluate_loss(projection, test_loader)
print('Test metrics:', test_metrics)

torch.save(unwrap_model(projection).state_dict(), SAVE_DIR / 'projection_head_state_dict.pt')
pd.DataFrame([test_metrics]).to_csv(SAVE_DIR / 'test_metrics.csv', index=False)
print('Saved artifacts to:', SAVE_DIR)


## Stage 2 - Light Fine-Tuning of Final Student Encoder Layers

Stage 2 starts from the best Stage 1 checkpoint. It unfreezes only a few final layers of `AITeamVN/Vietnamese_Embedding_v2` and uses a very small learning rate for the encoder, while the projection head is updated with a larger learning rate.

The goal of Stage 2 is to improve fashion-domain retrieval while reducing the risk of catastrophic forgetting.


In [ ]:
# =========================
# STAGE 2 CONFIG
# =========================
RUN_STAGE2 = True

STAGE2_UNFREEZE_LAST_N = 2
STAGE2_EPOCHS = 10
STAGE2_BATCH_SIZE = 128
STAGE2_ENCODER_LR = 2e-6
STAGE2_PROJECTION_LR = 1e-4
STAGE2_WEIGHT_DECAY = 1e-2
STAGE2_WARMUP_RATIO = 0.06
STAGE2_EARLY_STOPPING_PATIENCE = 3
STAGE2_GRAD_CLIP_NORM = 1.0

STAGE2_SAVE_DIR = SAVE_DIR / 'stage2_last_layers'
STAGE2_SAVE_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_BEST_PATH = STAGE2_SAVE_DIR / 'best_stage2_model.pt'

print('Stage 2 save dir:', STAGE2_SAVE_DIR)
print('Unfreeze last encoder layers:', STAGE2_UNFREEZE_LAST_N)


In [ ]:
class Stage2TextDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, teacher_embeds, weights):
        self.texts = dataframe['description_vi'].astype(str).tolist()
        self.teacher_embeds = teacher_embeds.float()
        self.weights = weights.float()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.teacher_embeds[idx], self.weights[idx]


def stage2_collate_fn(batch):
    texts, teacher_embeds, weights = zip(*batch)
    tokenized = student_tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=STUDENT_MAX_LENGTH,
        return_tensors='pt',
    )
    return tokenized, torch.stack(teacher_embeds), torch.tensor(weights, dtype=torch.float32)


class Stage2StudentProjection(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        projected = self.projection_head(pooled)
        return projected


def get_transformer_layers(encoder):
    enc = unwrap_model(encoder)
    candidates = [
        ('encoder.layer', lambda m: m.encoder.layer),
        ('roberta.encoder.layer', lambda m: m.roberta.encoder.layer),
        ('bert.encoder.layer', lambda m: m.bert.encoder.layer),
        ('base_model.encoder.layer', lambda m: m.base_model.encoder.layer),
    ]
    for name, getter in candidates:
        try:
            layers = getter(enc)
            if len(layers) > 0:
                print('Found transformer layers at:', name, '| count:', len(layers))
                return layers
        except Exception:
            pass
    raise AttributeError('Could not find transformer layers to unfreeze for this encoder.')


def prepare_stage2_model():
    # Start from the best Stage 1 projection checkpoint.
    checkpoint = torch.load(best_path, map_location='cpu')
    unwrap_model(projection).load_state_dict(checkpoint['projection_state_dict'])

    encoder_core = unwrap_model(student_encoder)
    projection_core = unwrap_model(projection)

    for p in encoder_core.parameters():
        p.requires_grad = False
    for p in projection_core.parameters():
        p.requires_grad = True

    layers = get_transformer_layers(encoder_core)
    for layer in layers[-STAGE2_UNFREEZE_LAST_N:]:
        for p in layer.parameters():
            p.requires_grad = True

    model_core = Stage2StudentProjection(encoder_core, projection_core).to(device)
    model = nn.DataParallel(model_core) if USE_DATA_PARALLEL else model_core

    encoder_trainable = sum(p.numel() for p in model_core.encoder.parameters() if p.requires_grad)
    projection_trainable = sum(p.numel() for p in model_core.projection_head.parameters() if p.requires_grad)
    print('Stage 2 trainable encoder params:', f'{encoder_trainable:,}')
    print('Stage 2 trainable projection params:', f'{projection_trainable:,}')
    return model, model_core


def build_stage2_loaders():
    train_ds_stage2 = Stage2TextDataset(train_df, train_teacher, train_weights)
    val_ds_stage2 = Stage2TextDataset(val_df, val_teacher, val_weights)
    test_ds_stage2 = Stage2TextDataset(test_df, test_teacher, test_weights)

    train_loader_stage2 = DataLoader(
        train_ds_stage2,
        batch_size=STAGE2_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        collate_fn=stage2_collate_fn,
    )
    val_loader_stage2 = DataLoader(
        val_ds_stage2,
        batch_size=STAGE2_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=stage2_collate_fn,
    )
    test_loader_stage2 = DataLoader(
        test_ds_stage2,
        batch_size=STAGE2_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=stage2_collate_fn,
    )
    return train_loader_stage2, val_loader_stage2, test_loader_stage2


In [ ]:
@torch.no_grad()
def evaluate_stage2_loss(model, loader):
    model.eval()
    total, total_align, total_struct, total_contr, n = 0.0, 0.0, 0.0, 0.0, 0
    for tokenized, teacher_embeds, weights in loader:
        tokenized = {k: v.to(device, non_blocking=True) for k, v in tokenized.items()}
        teacher_embeds = teacher_embeds.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)
        projected = model(input_ids=tokenized['input_ids'], attention_mask=tokenized['attention_mask'])
        loss, align, struct, contr = total_loss_fn(projected, teacher_embeds, weights)
        bs = teacher_embeds.size(0)
        total += loss.item() * bs
        total_align += align.item() * bs
        total_struct += struct.item() * bs
        total_contr += contr.item() * bs
        n += bs
    return {'loss': total / n, 'align': total_align / n, 'struct': total_struct / n, 'contrastive': total_contr / n}


if RUN_STAGE2:
    stage2_model, stage2_core = prepare_stage2_model()
    train_loader_stage2, val_loader_stage2, test_loader_stage2 = build_stage2_loaders()

    encoder_params = [p for p in stage2_core.encoder.parameters() if p.requires_grad]
    projection_params = [p for p in stage2_core.projection_head.parameters() if p.requires_grad]
    optimizer_stage2 = AdamW(
        [
            {'params': encoder_params, 'lr': STAGE2_ENCODER_LR},
            {'params': projection_params, 'lr': STAGE2_PROJECTION_LR},
        ],
        weight_decay=STAGE2_WEIGHT_DECAY,
    )
    total_steps_stage2 = STAGE2_EPOCHS * len(train_loader_stage2)
    warmup_steps_stage2 = int(STAGE2_WARMUP_RATIO * total_steps_stage2)
    scheduler_stage2 = get_linear_schedule_with_warmup(optimizer_stage2, warmup_steps_stage2, total_steps_stage2)
    scaler_stage2 = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_stage2_val = float('inf')
    stage2_patience = 0
    stage2_history = []

    for epoch in range(1, STAGE2_EPOCHS + 1):
        stage2_model.train()
        running_loss, running_align, running_struct, running_contr, seen = 0.0, 0.0, 0.0, 0.0, 0
        pbar = tqdm(train_loader_stage2, desc=f'Stage2 Epoch {epoch}/{STAGE2_EPOCHS}')

        for tokenized, teacher_embeds, weights in pbar:
            tokenized = {k: v.to(device, non_blocking=True) for k, v in tokenized.items()}
            teacher_embeds = teacher_embeds.to(device, non_blocking=True)
            weights = weights.to(device, non_blocking=True)

            optimizer_stage2.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                projected = stage2_model(input_ids=tokenized['input_ids'], attention_mask=tokenized['attention_mask'])
                loss, align, struct, contr = total_loss_fn(projected, teacher_embeds, weights)

            scaler_stage2.scale(loss).backward()
            scaler_stage2.unscale_(optimizer_stage2)
            torch.nn.utils.clip_grad_norm_(stage2_core.parameters(), STAGE2_GRAD_CLIP_NORM)
            scaler_stage2.step(optimizer_stage2)
            scaler_stage2.update()
            scheduler_stage2.step()

            bs = teacher_embeds.size(0)
            running_loss += loss.item() * bs
            running_align += align.item() * bs
            running_struct += struct.item() * bs
            running_contr += contr.item() * bs
            seen += bs
            pbar.set_postfix(
                loss=running_loss / seen,
                align=running_align / seen,
                struct=running_struct / seen,
                contr=running_contr / seen,
            )

        train_metrics = {
            'loss': running_loss / seen,
            'align': running_align / seen,
            'struct': running_struct / seen,
            'contrastive': running_contr / seen,
        }
        val_metrics = evaluate_stage2_loss(stage2_model, val_loader_stage2)
        stage2_history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})
        pd.DataFrame(stage2_history).to_csv(STAGE2_SAVE_DIR / 'stage2_training_history.csv', index=False)

        print(
            f"Stage2 Epoch {epoch}: train_loss={train_metrics['loss']:.5f} "
            f"val_loss={val_metrics['loss']:.5f} "
            f"val_align={val_metrics['align']:.6f} "
            f"val_struct={val_metrics['struct']:.6f} "
            f"val_contr={val_metrics['contrastive']:.5f}"
        )

        if val_metrics['loss'] < best_stage2_val:
            best_stage2_val = val_metrics['loss']
            stage2_patience = 0
            torch.save({
                'encoder_state_dict': stage2_core.encoder.state_dict(),
                'projection_state_dict': stage2_core.projection_head.state_dict(),
                'student_model_name': STUDENT_MODEL_NAME,
                'teacher_model_name': TEACHER_MODEL_NAME,
                'student_dim': STUDENT_DIM,
                'teacher_dim': TEACHER_DIM,
                'stage2_config': {
                    'unfreeze_last_n': STAGE2_UNFREEZE_LAST_N,
                    'encoder_lr': STAGE2_ENCODER_LR,
                    'projection_lr': STAGE2_PROJECTION_LR,
                    'batch_size': STAGE2_BATCH_SIZE,
                    'epochs': STAGE2_EPOCHS,
                    'lambda_align': LAMBDA_ALIGN,
                    'beta_struct': BETA_STRUCT,
                    'gamma_contrastive': GAMMA_CONTRASTIVE,
                    'contrastive_temperature': CONTRASTIVE_TEMPERATURE,
                },
            }, STAGE2_BEST_PATH)
            print('Saved best Stage 2 checkpoint:', STAGE2_BEST_PATH)
        else:
            stage2_patience += 1
            print(f'Stage 2 early stopping counter: {stage2_patience}/{STAGE2_EARLY_STOPPING_PATIENCE}')
            if stage2_patience >= STAGE2_EARLY_STOPPING_PATIENCE:
                print('Stage 2 early stopping triggered.')
                break

    stage2_checkpoint = torch.load(STAGE2_BEST_PATH, map_location=device)
    stage2_core.encoder.load_state_dict(stage2_checkpoint['encoder_state_dict'])
    stage2_core.projection_head.load_state_dict(stage2_checkpoint['projection_state_dict'])
    test_stage2_metrics = evaluate_stage2_loss(stage2_model, test_loader_stage2)
    print('Stage 2 test metrics:', test_stage2_metrics)
    pd.DataFrame([test_stage2_metrics]).to_csv(STAGE2_SAVE_DIR / 'stage2_test_metrics.csv', index=False)
else:
    print('RUN_STAGE2 is False. Skipping Stage 2.')


## Evaluation on Three Benchmarks

This section evaluates the student model and the FashionCLIP teacher on three prepared benchmark datasets: DeepFashion, Fashion200K, and KAGL. Results are saved as CSV files for comparison and reporting.


In [ ]:
# =========================
# 3-BENCHMARK EVALUATION CONFIG
# =========================
EVAL_OUTPUT_DIR = SAVE_DIR / 'benchmark_eval'
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_BATCH_SIZE_BENCHMARK = 128
EVALUATE_TEACHER_UPPER_BOUND = True
PREFER_STAGE2_CHECKPOINT_FOR_EVAL = True

BENCHMARKS = [
    {
        'name': 'DeepFashion',
        'dataset_root': '/kaggle/input/datasets/qucvinhchu/deepfashion-benchmark',
        'image_folder': 'deepfashion_images',
        'csv_file': 'deepfashion_benchmark_5k.csv',
    },
    {
        'name': 'Fashion200K',
        'dataset_root': '/kaggle/input/datasets/qucvinhchu/fashion200k-benchmark',
        'image_folder': 'fashion200k_images',
        'csv_file': 'fashion200k_benchmark_5k.csv',
    },
    {
        'name': 'KAGL',
        'dataset_root': '/kaggle/input/datasets/qucvinhchu/kagl-benchmark-5k',
        'image_folder': 'kagl_images',
        'csv_file': 'kagl_benchmark_5k.csv',
    },
]

for b in BENCHMARKS:
    b['image_dir'] = os.path.join(b['dataset_root'], b['image_folder'])
    b['csv_path'] = os.path.join(b['dataset_root'], b['csv_file'])
    if os.path.exists(b['csv_path']):
        b['df'] = pd.read_csv(b['csv_path'], encoding='utf-8-sig')
        print(f"Loaded {b['name']}: {len(b['df'])} rows")
    else:
        b['df'] = None
        print(f"Missing benchmark CSV: {b['csv_path']}")


In [ ]:
def build_evaluation_student_model(prefer_stage2=True):
    encoder = AutoModel.from_pretrained(STUDENT_MODEL_NAME)
    projection_head = ProjectionHead(
        encoder.config.hidden_size,
        TEACHER_DIM,
        hidden_dim=PROJECTION_HIDDEN_DIM,
        num_layers=PROJECTION_NUM_LAYERS,
        dropout=PROJECTION_DROPOUT,
    )

    loaded_checkpoint = None
    if prefer_stage2 and 'STAGE2_BEST_PATH' in globals() and STAGE2_BEST_PATH.exists():
        ckpt = torch.load(STAGE2_BEST_PATH, map_location='cpu')
        encoder.load_state_dict(ckpt['encoder_state_dict'])
        projection_head.load_state_dict(ckpt['projection_state_dict'])
        loaded_checkpoint = str(STAGE2_BEST_PATH)
    else:
        ckpt = torch.load(best_path, map_location='cpu')
        projection_head.load_state_dict(ckpt['projection_state_dict'])
        loaded_checkpoint = str(best_path)

    model = Stage2StudentProjection(encoder, projection_head).to(device).eval()
    if USE_DATA_PARALLEL:
        model = nn.DataParallel(model)
    print('Evaluation student checkpoint:', loaded_checkpoint)
    return model


eval_student_model = build_evaluation_student_model(PREFER_STAGE2_CHECKPOINT_FOR_EVAL)

vision_eval_model = CLIPVisionModelWithProjection.from_pretrained(TEACHER_MODEL_NAME).to(device).eval()
vision_eval_processor = CLIPProcessor.from_pretrained(TEACHER_MODEL_NAME)
for p in vision_eval_model.parameters():
    p.requires_grad = False

if EVALUATE_TEACHER_UPPER_BOUND:
    teacher_eval_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
    teacher_eval_text_model = CLIPTextModelWithProjection.from_pretrained(TEACHER_MODEL_NAME).to(device).eval()
    for p in teacher_eval_text_model.parameters():
        p.requires_grad = False


In [ ]:
def resolve_image_path(image_dir, raw_path):
    return os.path.join(image_dir, os.path.basename(str(raw_path)))


def prepare_valid_rows(df_benchmark, image_dir):
    rows = []
    image_paths = []
    for _, row in df_benchmark.iterrows():
        image_path = resolve_image_path(image_dir, row['image_paths'])
        if os.path.exists(image_path):
            rows.append(row)
            image_paths.append(image_path)
    valid_df = pd.DataFrame(rows).reset_index(drop=True)
    return valid_df, image_paths


@torch.no_grad()
def encode_benchmark_images(image_paths, batch_size=EVAL_BATCH_SIZE_BENCHMARK):
    all_embeds = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='Encode images'):
        batch_paths = image_paths[i:i + batch_size]
        images = [Image.open(p).convert('RGB') for p in batch_paths]
        inputs = vision_eval_processor(images=images, return_tensors='pt').to(device)
        embeds = vision_eval_model(**inputs).image_embeds
        embeds = F.normalize(embeds, p=2, dim=-1)
        all_embeds.append(embeds.cpu())
    return torch.cat(all_embeds, dim=0)


@torch.no_grad()
def encode_student_vi_texts(texts, batch_size=EVAL_BATCH_SIZE_BENCHMARK):
    all_embeds = []
    raw_texts = [str(text) for text in texts]
    for i in tqdm(range(0, len(raw_texts), batch_size), desc='Encode student VI'):
        batch = raw_texts[i:i + batch_size]
        inputs = student_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=STUDENT_MAX_LENGTH,
            return_tensors='pt',
        ).to(device)
        embeds = eval_student_model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        embeds = F.normalize(embeds, p=2, dim=-1)
        all_embeds.append(embeds.cpu())
    return torch.cat(all_embeds, dim=0)


@torch.no_grad()
def encode_teacher_en_texts(texts, batch_size=EVAL_BATCH_SIZE_BENCHMARK):
    all_embeds = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encode teacher EN'):
        batch = [str(x) for x in texts[i:i + batch_size]]
        inputs = teacher_eval_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=TEACHER_MAX_LENGTH,
            return_tensors='pt',
        ).to(device)
        embeds = teacher_eval_text_model(**inputs).text_embeds
        embeds = F.normalize(embeds, p=2, dim=-1)
        all_embeds.append(embeds.cpu())
    return torch.cat(all_embeds, dim=0)


def compute_retrieval_metrics(text_embeds, image_embeds):
    sim = text_embeds @ image_embeds.T
    n = sim.shape[0]
    targets = torch.arange(n)

    t2i_sorted = sim.argsort(dim=1, descending=True)
    i2t_sorted = sim.T.argsort(dim=1, descending=True)

    t2i_ranks = (t2i_sorted == targets[:, None]).nonzero(as_tuple=True)[1] + 1
    i2t_ranks = (i2t_sorted == targets[:, None]).nonzero(as_tuple=True)[1] + 1

    metrics = {'Valid Pairs': int(n)}
    for k in [1, 5, 10]:
        metrics[f'T2I_R@{k}'] = (t2i_ranks <= k).float().mean().item() * 100
        metrics[f'I2T_R@{k}'] = (i2t_ranks <= k).float().mean().item() * 100
    metrics['T2I_MedR'] = float(torch.median(t2i_ranks.float()).item())
    metrics['I2T_MedR'] = float(torch.median(i2t_ranks.float()).item())
    metrics['T2I_MeanR'] = float(torch.mean(t2i_ranks.float()).item())
    metrics['I2T_MeanR'] = float(torch.mean(i2t_ranks.float()).item())
    return metrics


def print_metrics_table(title, results):
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)
    df_results = pd.DataFrame(results).T
    df_results.index.name = 'Benchmark'
    display(df_results)
    return df_results


In [ ]:
student_results = {}
teacher_results = {}

for b in BENCHMARKS:
    if b['df'] is None:
        continue
    print('\n' + '=' * 80)
    print('Evaluating benchmark:', b['name'])
    print('=' * 80)

    valid_df, valid_image_paths = prepare_valid_rows(b['df'], b['image_dir'])
    print('Valid image-text pairs:', len(valid_df))
    if len(valid_df) == 0:
        continue

    image_embeds_eval = encode_benchmark_images(valid_image_paths)

    vi_texts = valid_df['description_vi'].astype(str).tolist()
    student_text_embeds = encode_student_vi_texts(vi_texts)
    student_results[b['name']] = compute_retrieval_metrics(student_text_embeds, image_embeds_eval)

    if EVALUATE_TEACHER_UPPER_BOUND:
        en_texts = valid_df['description_en'].astype(str).tolist()
        teacher_text_embeds = encode_teacher_en_texts(en_texts)
        teacher_results[b['name']] = compute_retrieval_metrics(teacher_text_embeds, image_embeds_eval)

student_df = print_metrics_table('STUDENT VI RESULTS', student_results)
student_df.to_csv(EVAL_OUTPUT_DIR / 'student_3bench_results.csv', encoding='utf-8-sig')

if EVALUATE_TEACHER_UPPER_BOUND and teacher_results:
    teacher_df = print_metrics_table('TEACHER EN UPPER BOUND RESULTS', teacher_results)
    teacher_df.to_csv(EVAL_OUTPUT_DIR / 'teacher_3bench_results.csv', encoding='utf-8-sig')

    gap_rows = {}
    for name in student_results:
        if name not in teacher_results:
            continue
        gap_rows[name] = {
            metric: student_results[name][metric] - teacher_results[name][metric]
            for metric in ['T2I_R@1', 'T2I_R@5', 'T2I_R@10', 'I2T_R@1', 'I2T_R@5', 'I2T_R@10']
        }
    gap_df = print_metrics_table('STUDENT - TEACHER GAP', gap_rows)
    gap_df.to_csv(EVAL_OUTPUT_DIR / 'student_teacher_gap_3bench.csv', encoding='utf-8-sig')

print('Saved benchmark outputs to:', EVAL_OUTPUT_DIR)
